# 01 — Baselines

Score buy-and-hold, 60-minute momentum, 60-minute mean reversion, and a long volatility-targeted position on the development sample. A position at time t earns the next exact one-minute return. These runs are benchmarks, not `EXP-H01` evidence, and they do not read the sealed test.

Run `00_data_validation.ipynb` first so the dataset manifest exists.


In [1]:
from pathlib import Path
import os
import sys

root = Path.cwd().resolve()
while not (root / "research" / "__init__.py").is_file() and root != root.parent:
    root = root.parent
if not (root / "research" / "__init__.py").is_file():
    raise RuntimeError("open this notebook from the trading repository")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["AWS_PROFILE"] = "trading"
os.environ["AWS_DEFAULT_REGION"] = "ap-southeast-2"
os.environ["DATA_STORE"] = "s3"
os.environ["S3_BUCKET"] = "trading-data-centheos"


In [2]:
import pandas as pd

from research.evaluate import align_to_frozen_dataset
from research.load import load_research_bars
from research.manifest import load_manifest
from research.protocol import H1_CONFIG

manifest_path = root / "research" / "registry" / "datasets" / "btcusdt-1m.json"
frozen = load_manifest(manifest_path)
bars, objects = load_research_bars(
    venue=H1_CONFIG["venue"],
    symbol=H1_CONFIG["symbol"],
    timeframe=H1_CONFIG["timeframe"],
    bucket=H1_CONFIG["bucket"],
    profile=H1_CONFIG["profile"],
    region=H1_CONFIG["region"],
)
if bars.empty:
    raise RuntimeError("BTCUSDT one-minute load returned no rows")
kept, report, appended = align_to_frozen_dataset(bars, frozen, objects)
print(
    f"frozen digest={frozen.digest} rows={report.rows_out} "
    f"collector_rows_after_freeze={appended}"
)


frozen digest=d67871b2eebd796cb37c7cafa89677fbca4b31ad0bdae959a5f0e2efeecc6c88 rows=3585983 collector_rows_after_freeze=0


In [3]:
from research.evaluate import evaluate_baselines
from research.manifest import load_manifest
from research.protocol import BASELINE_CONFIG
from research.registry import config_digest

manifest = load_manifest(root / "research" / "registry" / "datasets" / "btcusdt-1m.json")
if manifest.symbol != "BTCUSDT" or manifest.timeframe != "1m":
    raise RuntimeError(f"baseline notebook expected BTCUSDT 1m, found {manifest.symbol} {manifest.timeframe}")
reloaded = load_manifest(manifest_path)
if reloaded.digest != manifest.digest:
    raise RuntimeError("dataset manifest changed while this notebook was running")
print("baseline configuration", config_digest(BASELINE_CONFIG))
print("sealed test is not read by evaluate_baselines")
evaluate_baselines(kept["close"], BASELINE_CONFIG)


baseline configuration fbb51e2055b84928fcb202fffd215c32eb96fc755c7bdc833f05ec1996e441a9
sealed test is not read by evaluate_baselines


Excluded 1 close rows from one-minute baseline returns
Excluded 1 close rows from one-minute baseline returns
Baseline exclusions: nonconsecutive_returns=1 missing_volatility_scale=0


,baseline,sealed_access,observations,total_simple_return,sharpe,max_drawdown,average_turnover,average_absolute_position
0,buy_and_hold,False,3059998,14.524487,1.029844,-0.775836,3.267976e-07,1.000000
1,momentum,False,3059998,-1.000000,-49.776651,-1.000000,1.311514e-01,0.999635
2,mean_reversion,False,3059998,-1.000000,-47.430969,-1.000000,1.311514e-01,0.999635
3,volatility_targeted_long,False,3059998,-0.955832,-1.039771,-0.974033,3.332611e-03,0.873680
